# Experiment 5 · Cross-Experiment Analysis

Clean, self-contained analysis notebook for **MTH 5320 — Project 1**.
Loads trained checkpoints from Experiments 1–4 and evaluates them on the
held-out clean test set with consistent metrics, visualizations, and a
fixed nearest-neighbour Bayes-error proxy.

**Does not train any model.** Run `04_Experiment_4.ipynb` first if you
want the Exp 4 (Spherical) row to appear in the comparison table.


> **How to run (Google Colab):** Upload this notebook. Runtime → Change runtime
> type → **T4 GPU**. Upload `L058.txt` and any `.pt` checkpoint files you want
> evaluated, then **Runtime → Run all**.


## 0 · Environment Setup

In [ ]:
# ============================================================
# Environment Setup — run this first
# ============================================================
# Colab ships with PyTorch and sklearn pre-installed.
# Optuna and tqdm are the only additions needed.
!pip install optuna tqdm --quiet

# Sensor location file (L058.txt):
#   - If it is already next to this notebook (e.g. a cloned repo), use it.
#   - Otherwise, on Colab, prompt for an upload.
import os
if os.path.exists('L058.txt'):
    print('Found L058.txt in the working directory \u2014 no upload needed.')
else:
    try:
        from google.colab import files
        uploaded = files.upload()  # prompts for L058.txt
    except ImportError:
        raise FileNotFoundError(
            'L058.txt not found. Place it next to this notebook (or upload it on Colab).'
        )


## 1 · Shared Infrastructure

### 1.1 Magnetometer Time Series Simulator (MTSS)

In [ ]:
"""
Magnetometer Time Series Simulator for Magnetic Localization Research

This module provides a high-fidelity simulator for generating synthetic
magnetometer data from dipole sources, designed for training neural networks
for magnetic source localization.

Author: Strider Settgast with Deepseek
Date: 2026-06-01
Version: 1.0.0
"""

import numpy as np
from typing import Optional, Tuple, Dict, List
from dataclasses import dataclass
import warnings
from enum import Enum


class SourceType(Enum):
    """Types of magnetic sources that can be simulated."""
    DIPOLE = "dipole"
    MONOPOLE = "monopole"  # Not physically real but useful for testing
    #QUADRUPOLE = "quadrupole"


class NoiseModel(Enum):
    """Available noise models for sensor simulation."""
    GAUSSIAN = "gaussian"
    UNIFORM = "uniform"
    MIXED = "mixed"  # Combination of Gaussian and outliers


@dataclass
class SimulatorConfig:
    """
    Configuration class for the TimeSeriesSimulator.

    Attributes:
        sensor_positions: (n_sensors, 3) array of sensor coordinates
        n_sensors: Number of sensors (auto-detected from positions)
        magnetic_constant: mu_0/(4pi) in SI units (default 1e-7 for normalized)
        default_source_bounds: [min, max] for source position cube
        default_moment_range: [min, max] for magnetic moment magnitude
        noise_std: Standard deviation for Gaussian noise
        noise_outlier_fraction: Fraction of outliers for robust testing
        random_seed: For reproducibility
        normalize_outputs: Whether to normalize target positions
        output_bounds: Bounds for output normalization (if used)
    """
    sensor_positions: np.ndarray
    magnetic_constant: float = 1e-7  # mu_0/(4pi) in SI, or 1.0 for normalized
    default_source_bounds: Tuple[float, float] = (-5.0, 5.0)
    default_moment_range: Tuple[float, float] = (0.5, 2.0)
    noise_std: float = 0.01
    noise_outlier_fraction: float = 0.01
    random_seed: Optional[int] = 42
    normalize_outputs: bool = False
    output_bounds: Optional[Tuple[float, float]] = None

    def __post_init__(self):
        """Validate configuration after initialization."""
        self.n_sensors = len(self.sensor_positions)
        if self.sensor_positions.shape[1] != 3:
            raise ValueError(f"Sensor positions must have 3 columns, got {self.sensor_positions.shape[1]}")

        if self.random_seed is not None:
            np.random.seed(self.random_seed)

        if self.output_bounds is None:
            self.output_bounds = self.default_source_bounds


class TimeSeriesSimulator:
    """
    Industry-grade simulator for magnetometer time series data.

    This class generates realistic magnetic field readings from moving or static
    dipole sources, with configurable noise, sensor layouts, and source dynamics.

    Features:
        - Static and moving source simulation
        - Multiple noise models (Gaussian, uniform, mixed)
        - Sensor noise and dropout simulation
        - Temporal correlation for realistic time series
        - Batching and streaming data generation
        - Built-in visualization tools

    Examples:
        >>> # Initialize with sensor positions
        >>> sensor_positions = np.random.rand(29, 3) * 10
        >>> sim = TimeSeriesSimulator(sensor_positions)
        >>>
        >>> # Generate static source data
        >>> X, y = sim.generate_batch(n_samples=1000)
        >>>
        >>> # Generate moving source time series
        >>> trajectory = lambda t: np.array([np.sin(t), np.cos(t), t*0.1])
        >>> X_series, y_series = sim.generate_time_series(
        ...     n_timesteps=500,
        ...     trajectory_func=trajectory
        ... )
    """

    def __init__(self, sensor_positions: np.ndarray, config: Optional[SimulatorConfig] = None):
        """
        Initialize the simulator with sensor geometry.

        Args:
            sensor_positions: (n_sensors, 3) array of sensor coordinates in meters
            config: Optional SimulatorConfig object. If None, uses defaults.
        """
        self.sensor_positions = sensor_positions
        self.n_sensors = len(sensor_positions)

        if config is None:
            config = SimulatorConfig(sensor_positions=sensor_positions)
        else:
            config.sensor_positions = sensor_positions
            config.__post_init__()

        self.config = config
        self._validate_sensor_positions()

        # Pre-compute sensor positions for efficiency
        self.sensor_positions = config.sensor_positions
        self.n_sensors = config.n_sensors
        self.n_features = self.n_sensors * 3  # Bx, By, Bz for each sensor

        # Statistics for online normalization (if needed)
        self.input_mean = None
        self.input_std = None
        self.output_mean = None
        self.output_std = None

        print(f"✅ Simulator initialized: {self.n_sensors} sensors, {self.n_features} features")

    def _validate_sensor_positions(self):
        """Validate that sensor positions are physically reasonable."""
        if np.any(np.isnan(self.sensor_positions)):
            raise ValueError("Sensor positions contain NaN values")
        if np.any(np.isinf(self.sensor_positions)):
            raise ValueError("Sensor positions contain infinite values")

        # Check for duplicate sensors (within 1mm tolerance)
        unique_positions = np.unique(np.round(self.sensor_positions, decimals=6), axis=0)
        if len(unique_positions) < self.n_sensors:
            warnings.warn(f"Duplicate sensor positions detected. {self.n_sensors - len(unique_positions)} duplicates found.")

    def monopole_field(self,
                       sensor_pos: np.ndarray,
                       source_pos: np.ndarray,
                       magnetic_charge: float) -> np.ndarray:
        """
        Calculate magnetic field from a theoretical monopole (1/r^2 falloff).
        """
        r_vec = sensor_pos - source_pos
        r = np.linalg.norm(r_vec)
        if r < 1e-9:
            return np.zeros(3)

        r_hat = r_vec / r
        B = (r_hat) / (r**2)
        B *= self.config.magnetic_constant * magnetic_charge

        return B

    def dipole_field(self,
                     sensor_pos: np.ndarray,
                     source_pos: np.ndarray,
                     magnetic_moment: np.ndarray) -> np.ndarray:
        """
        Calculate magnetic field from a dipole source at a single sensor.

        Args:
            sensor_pos: (3,) array of sensor position [x, y, z]
            source_pos: (3,) array of source position [x, y, z]
            magnetic_moment: (3,) array of magnetic moment vector [mx, my, mz]

        Returns:
            (3,) array of magnetic field [Bx, By, Bz] in Tesla (or normalized units)

        Raises:
            ValueError: If inputs have incorrect shapes
        """
        # Input validation
        if sensor_pos.shape != (3,):
            raise ValueError(f"sensor_pos must be (3,), got {sensor_pos.shape}")
        if source_pos.shape != (3,):
            raise ValueError(f"source_pos must be (3,), got {source_pos.shape}")
        if magnetic_moment.shape != (3,):
            raise ValueError(f"magnetic_moment must be (3,), got {magnetic_moment.shape}")

        # Calculate relative position vector
        r_vec = sensor_pos - source_pos
        r = np.linalg.norm(r_vec)

        # Avoid division by zero (sensor at source location)
        if r < 1e-9:
            return np.zeros(3)

        r_hat = r_vec / r
        m_dot_r_hat = np.dot(magnetic_moment, r_hat)

        # Dipole field equation
        B = (3 * r_hat * m_dot_r_hat - magnetic_moment) / (r**3)

        # Apply magnetic constant (normalization)
        B *= self.config.magnetic_constant

        return B


    def generate_source(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate a random magnetic source (position and moment).

        Returns:
            Tuple of (source_position, magnetic_moment)

        Note:
            Position is uniform within source_bounds.
            Moment magnitude is uniform within moment_range.
            Moment direction is uniformly distributed on sphere.
        """
        # Random source position within bounds
        low, high = self.config.default_source_bounds
        source_pos = np.random.uniform(low, high, size=3)

        # Random magnetic moment magnitude
        mag_low, mag_high = self.config.default_moment_range
        moment_magnitude = np.random.uniform(mag_low, mag_high)

        # Random direction on sphere (uniform distribution)
        direction = np.random.randn(3)
        direction /= np.linalg.norm(direction)

        magnetic_moment = direction * moment_magnitude

        return source_pos, magnetic_moment

    def compute_fields(self, source_pos: np.ndarray, magnetic_moment: np.ndarray, source_type: SourceType = SourceType.DIPOLE) -> np.ndarray:
        """
        Compute magnetic fields at all sensors for a given source.

        Args:
            source_pos: (3,) array of source position
            magnetic_moment: (3,) array of magnetic moment
            source_type: The type of magnetic source (Enum)

        Returns:
            (n_features,) array of flattened sensor readings (Bx,By,Bz for each sensor)
        """
        fields = []
        for sensor_pos in self.sensor_positions:
            if source_type == SourceType.DIPOLE:
                B = self.dipole_field(sensor_pos, source_pos, magnetic_moment)
            elif source_type == SourceType.MONOPOLE:
                charge = np.linalg.norm(magnetic_moment)
                B = self.monopole_field(sensor_pos, source_pos, charge)

            fields.extend(B)

        return np.array(fields)

    def add_noise(self,
                  fields: np.ndarray,
                  noise_model: NoiseModel = NoiseModel.GAUSSIAN) -> np.ndarray:
        """
        Add realistic noise to sensor readings.

        Args:
            fields: Clean magnetic field readings
            noise_model: Type of noise to add

        Returns:
            Noisy field readings
        """
        noisy = fields.copy()

        if noise_model == NoiseModel.GAUSSIAN:
            noise = np.random.normal(0, self.config.noise_std, size=fields.shape)
            noisy += noise

        elif noise_model == NoiseModel.UNIFORM:
            noise_range = self.config.noise_std * np.sqrt(3)  # Match variance
            noise = np.random.uniform(-noise_range, noise_range, size=fields.shape)
            noisy += noise

        elif noise_model == NoiseModel.MIXED:
            # Gaussian noise
            noise = np.random.normal(0, self.config.noise_std, size=fields.shape)
            noisy += noise

            # Add outliers
            n_outliers = int(len(fields) * self.config.noise_outlier_fraction)
            outlier_indices = np.random.choice(len(fields), n_outliers, replace=False)
            outlier_magnitude = self.config.noise_std * 10
            noisy[outlier_indices] += np.random.normal(0, outlier_magnitude, size=n_outliers)

        return noisy

    def generate_sample(self,
                        source_pos: Optional[np.ndarray] = None,
                        magnetic_moment: Optional[np.ndarray] = None,
                        add_noise: bool = True,
                        noise_model: NoiseModel = NoiseModel.GAUSSIAN) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate a single training sample.

        Args:
            source_pos: Optional source position (random if None)
            magnetic_moment: Optional magnetic moment (random if None)
            add_noise: Whether to add noise to the readings
            noise_model: Which noise model to use

        Returns:
            Tuple of (features, target) where:
                features: (n_features,) array of magnetic field readings
                target: (3,) array of source position
        """
        random_pos, random_moment = self.generate_source()

        if source_pos is None:
            source_pos = random_pos
        if magnetic_moment is None:
            magnetic_moment = random_moment

        fields = self.compute_fields(source_pos, magnetic_moment)

        if add_noise:
            fields = self.add_noise(fields, noise_model)

        target = source_pos.copy()

        # Optional output normalization
        if self.config.normalize_outputs:
            low, high = self.config.output_bounds
            target = (target - low) / (high - low) * 2 - 1  # Normalize to [-1, 1]

        return fields, target

    def generate_batch(self,
                       n_samples: int,
                       add_noise: bool = True,
                       noise_model: NoiseModel = NoiseModel.GAUSSIAN) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate a batch of training samples.

        Args:
            n_samples: Number of samples to generate
            add_noise: Whether to add noise
            noise_model: Noise model to use

        Returns:
            Tuple of (X, y) where:
                X: (n_samples, n_features) array of features
                y: (n_samples, 3) array of targets
        """
        X = np.zeros((n_samples, self.n_features))
        y = np.zeros((n_samples, 3))

        for i in range(n_samples):
            X[i], y[i] = self.generate_sample(add_noise=add_noise, noise_model=noise_model)

        return X, y

    def generate_time_series(self,
                            n_timesteps: int,
                            trajectory_func: Optional[callable] = None,
                            moment_func: Optional[callable] = None,
                            dt: float = 0.01,
                            add_noise: bool = True,
                            noise_model: NoiseModel = NoiseModel.GAUSSIAN) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate a time series with moving source.

        Args:
            n_timesteps: Number of time steps to generate
            trajectory_func: Function t -> (x, y, z) position.
                            If None, uses random walk.
            moment_func: Function t -> (mx, my, mz) magnetic moment.
                        If None, uses constant random moment.
            dt: Time step duration in seconds
            add_noise: Whether to add noise
            noise_model: Noise model to use

        Returns:
            Tuple of (X, y) where:
                X: (n_timesteps, n_features) time series of features
                y: (n_timesteps, 3) time series of source positions

        Examples:
            >>> # Linear motion
            >>> def linear_motion(t):
            ...     return np.array([t, 0, 0])
            >>> X, y = sim.generate_time_series(1000, trajectory_func=linear_motion)

            >>> # Circular motion with sinusoidal moment
            >>> def circular_motion(t):
            ...     return np.array([np.sin(t), np.cos(t), 0])
            >>> X, y = sim.generate_time_series(1000, trajectory_func=circular_motion)
        """
        times = np.arange(n_timesteps) * dt

        if trajectory_func is None:
            # Random walk trajectory
            positions = np.zeros((n_timesteps, 3))
            positions[0] = np.random.uniform(*self.config.default_source_bounds, size=3)
            step_size = 0.1
            for t in range(1, n_timesteps):
                positions[t] = positions[t-1] + np.random.randn(3) * step_size
                # Keep within bounds (soft boundary with reflection)
                low, high = self.config.default_source_bounds
                for dim in range(3):
                    if positions[t, dim] < low:
                        positions[t, dim] = low + (low - positions[t, dim])
                    elif positions[t, dim] > high:
                        positions[t, dim] = high - (positions[t, dim] - high)
        else:
            positions = np.array([trajectory_func(t) for t in times])

        if moment_func is None:
            # Constant random moment
            _, magnetic_moment = self.generate_source()
            moments = np.tile(magnetic_moment, (n_timesteps, 1))
        else:
            moments = np.array([moment_func(t) for t in times])

        # Generate fields for each time step
        X = np.zeros((n_timesteps, self.n_features))
        y = positions.copy()

        for i in range(n_timesteps):
            fields = self.compute_fields(positions[i], moments[i])
            if add_noise:
                fields = self.add_noise(fields, noise_model)
            X[i] = fields

        return X, y

    def compute_snr(self, clean_fields: np.ndarray, noisy_fields: np.ndarray) -> float:
        """
        Compute Signal-to-Noise Ratio for generated data.

        Args:
            clean_fields: Clean magnetic field readings
            noisy_fields: Noisy readings

        Returns:
            SNR in decibels (dB)
        """
        signal_power = np.mean(clean_fields ** 2)
        noise_power = np.mean((clean_fields - noisy_fields) ** 2)

        if noise_power < 1e-12:
            return float('inf')

        snr = 10 * np.log10(signal_power / noise_power)
        return snr

    def get_data_statistics(self, X: np.ndarray, y: np.ndarray) -> Dict[str, Dict[str, float]]:
        """
        Compute comprehensive statistics for generated data.

        Args:
            X: Features array (n_samples, n_features)
            y: Targets array (n_samples, 3)

        Returns:
            Dictionary with statistics for inputs and outputs
        """
        stats = {
            'input': {
                'mean': np.mean(X, axis=0).tolist(),
                'std': np.std(X, axis=0).tolist(),
                'min': np.min(X, axis=0).tolist(),
                'max': np.max(X, axis=0).tolist(),
                'has_nan': np.any(np.isnan(X)),
                'has_inf': np.any(np.isinf(X))
            },
            'output': {
                'mean': np.mean(y, axis=0).tolist(),
                'std': np.std(y, axis=0).tolist(),
                'min': np.min(y, axis=0).tolist(),
                'max': np.max(y, axis=0).tolist(),
                'has_nan': np.any(np.isnan(y)),
                'has_inf': np.any(np.isinf(y))
            }
        }
        return stats

    def visualize_sensor_array(self, ax=None):
        """
        Visualize the 3D positions of sensors.

        Args:
            ax: Matplotlib 3D axis (creates new if None)
        """
        try:
            import matplotlib.pyplot as plt
            from mpl_toolkits.mplot3d import Axes3D

            if ax is None:
                fig = plt.figure(figsize=(10, 8))
                ax = fig.add_subplot(111, projection='3d')

            ax.scatter(self.sensor_positions[:, 0],
                      self.sensor_positions[:, 1],
                      self.sensor_positions[:, 2],
                      c='red', marker='o', s=50, label='Sensors')

            ax.set_xlabel('X (m)')
            ax.set_ylabel('Y (m)')
            ax.set_zlabel('Z (m)')
            ax.set_title(f'Sensor Array Layout ({self.n_sensors} Sensors)')
            ax.legend()

            # Set equal aspect ratio
            max_range = np.max([
                np.ptp(self.sensor_positions[:, 0]),
                np.ptp(self.sensor_positions[:, 1]),
                np.ptp(self.sensor_positions[:, 2])
            ]) / 2.0
            mid_x = np.mean(self.sensor_positions[:, 0])
            mid_y = np.mean(self.sensor_positions[:, 1])
            mid_z = np.mean(self.sensor_positions[:, 2])
            ax.set_xlim(mid_x - max_range, mid_x + max_range)
            ax.set_ylim(mid_y - max_range, mid_y + max_range)
            ax.set_zlim(mid_z - max_range, mid_z + max_range)

            return ax

        except ImportError:
            print("Matplotlib not available. Install with: pip install matplotlib")
            return None

    def save_dataset(self, X: np.ndarray, y: np.ndarray, filepath: str):
        """
        Save generated dataset to disk (NPZ format).

        Args:
            X: Features array
            y: Targets array
            filepath: Path to save file (.npz extension recommended)
        """
        np.savez_compressed(filepath, X=X, y=y,
                           sensor_positions=self.sensor_positions,
                           config=vars(self.config))
        print(f"✅ Dataset saved to {filepath}")

    @classmethod
    def load_dataset(cls, filepath: str):
        """
        Load a saved dataset.

        Args:
            filepath: Path to saved .npz file

        Returns:
            Dictionary containing X, y, sensor_positions, and config
        """
        data = np.load(filepath, allow_pickle=True)
        return {
            'X': data['X'],
            'y': data['y'],
            'sensor_positions': data['sensor_positions'],
            'config': data['config'].item() if 'config' in data else None
        }

### 1.2 Sensor Array Loader (`L058.txt` → 29 stations)

In [ ]:
import numpy as np
import pandas as pd


## Load the sensor locations

# Define column names
column_names = ['Station', 'Lshell', 'M_Lat', 'M_Lon', 'G_lat', 'G_lon']

# Read the file using read_fwf (fixed-width format)
locations_df = pd.read_fwf('L058.txt', names=column_names, skiprows=1)  # skiprows=1 to skip the "029" line

# Sort by station name
locations_df.sort_values(by='Station', inplace=True)

# --- 3D Cartesian Conversion ---
# Mean radius of Earth in kilometers (use 1.0 for a normalized unit sphere)
R = 6371.0

# Convert geographic latitude and longitude to radians
lat_rad = np.radians(locations_df["G_lat"])
lon_rad = np.radians(locations_df["G_lon"])

# Compute X, Y, and Z coordinates
locations_df["X"] = R * np.cos(lat_rad) * np.cos(lon_rad)
locations_df["Y"] = R * np.cos(lat_rad) * np.sin(lon_rad)
locations_df["Z"] = R * np.sin(lat_rad)

# Extract X, Y, Z columns into a 2D NumPy array
sensors_xyz_locations = locations_df[["X", "Y", "Z"]].to_numpy()

# Check the shape of your new array
print(f"\n \n✓ Loaded: - Sensor Locations {sensors_xyz_locations.shape[0]} rows × {sensors_xyz_locations.shape[1]} columns")
print(f"\nFirst 5 rows (preview):")
print(sensors_xyz_locations[:5, :5])

### 1.3 Multi-Task Neural Network (includes SphericalOutputLayer for Exp 4)

In [ ]:
"""
Multi-Task Neural Network for Magnetic Source Identification
Industry-standard architecture with residual connections, batch normalization,
and proper regularization for geophysical inversion problems.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Dict, Tuple, Optional, List
from dataclasses import dataclass
import optuna
from optuna.trial import Trial
import warnings


@dataclass
class ModelConfig:
    """Configuration for the multi-task magnetic source identification model."""
    # Architecture
    input_dim: int = 87  # 29 sensors × 3 components
    hidden_dims: List[int] = None  # Will be set in __post_init__
    dropout_rate: float = 0.3
    activation: str = "silu"  # SiLU/Swish often better for geophysical data

    # Task-specific heads
    loc_hidden_dims: List[int] = None  # Regression head
    type_hidden_dims: List[int] = None  # Classification head

    # Regularization
    weight_decay: float = 1e-5
    batch_norm_momentum: float = 0.1
    use_residual: bool = True

    # Training
    learning_rate: float = 1e-3
    batch_size: int = 64           # Mini-batch size (tunable by Optuna)
    loc_loss_weight: float = 1.0  # Weight for position loss
    type_loss_weight: float = 1.0  # Weight for type classification loss

    def __post_init__(self):
        if self.hidden_dims is None:
            self.hidden_dims = [256, 256, 128]
        if self.loc_hidden_dims is None:
            self.loc_hidden_dims = [64, 32]
        if self.type_hidden_dims is None:
            self.type_hidden_dims = [64, 32]


class ResidualBlock(nn.Module):
    """Residual block with batch normalization and dropout.

    Uses a projection shortcut so the block works even when its input and
    output dimensions differ (standard ResNet practice). Each block also owns
    its own activation instances rather than sharing a single module passed in
    from the parent network.
    """

    @staticmethod
    def _make_activation(activation_type: str) -> nn.Module:
        if activation_type == "relu":
            return nn.ReLU()
        elif activation_type == "silu":
            return nn.SiLU()
        elif activation_type == "gelu":
            return nn.GELU()
        else:
            raise ValueError(f"Unsupported activation: {activation_type}")

    def __init__(self, in_dim: int, out_dim: int, dropout_rate: float,
                 activation_type: str, bn_momentum: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.bn1 = nn.BatchNorm1d(out_dim, momentum=bn_momentum)
        self.fc2 = nn.Linear(out_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim, momentum=bn_momentum)
        self.dropout = nn.Dropout(dropout_rate)

        # Project the residual when dimensions change so (x + residual) aligns.
        if in_dim != out_dim:
            self.shortcut = nn.Linear(in_dim, out_dim, bias=False)
        else:
            self.shortcut = nn.Identity()

        # Fresh activation instances owned by this block (no shared reference).
        self.activation1 = self._make_activation(activation_type)
        self.activation2 = self._make_activation(activation_type)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        x = self.activation1(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.bn2(self.fc2(x))
        x = self.activation2(x + residual)
        return x



# ─── Shell geometry constants ──────────────────────────────────────────────────
EARTH_RADIUS_KM = 6371.0
SHELL_MIN_KM    = EARTH_RADIUS_KM + 800    # 7,171 km  (min source altitude)
SHELL_RANGE_KM  = 3100.0 - 800.0           # 2,300 km  (altitude band width)


class SphericalOutputLayer(nn.Module):
    """
    Replaces the final nn.Linear(→3) of the regression head.

    Maps backbone features → a unit-sphere direction vector + shell radius →
    Cartesian km → MinMaxScaler [0,1] space (identical format to the
    original Cartesian head so the loss function needs no changes).

    Uses a direction-vector + radius parameterisation:
        direction = F.normalize(raw[:, :3])  — unit vector, no angular singularity
        r         = SHELL_MIN + sigmoid(raw[:, 3]) * SHELL_RANGE  — shell-constrained

    This is fully symmetric in X/Y/Z (no preferred axis) and avoids the
    sin(phi)=0 collapse of the original tanh·π longitude encoding, which
    produces Y=0 at both tanh's initialisation point (x=0) and its
    saturation limits (x→±∞).
    """

    def __init__(self, in_dim: int,
                 scaler_min: "array-like",
                 scaler_range: "array-like"):
        super().__init__()
        self.linear = nn.Linear(in_dim, 4)   # (dir_x, dir_y, dir_z, r_raw)
        self.register_buffer(
            'scaler_min',   torch.as_tensor(scaler_min,   dtype=torch.float32))
        self.register_buffer(
            'scaler_range', torch.as_tensor(scaler_range, dtype=torch.float32))

    def reset_parameters(self):
        """Re-initialise with gain=1.0 so ||dir_raw|| starts at a moderate scale.

        Called from MultiTaskMagneticNet after the global xavier(gain=0.5) sweep,
        which would leave direction raw-outputs so small that F.normalize
        amplifies gradients unnecessarily.
        """
        nn.init.xavier_normal_(self.linear.weight, gain=1.0)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raw       = self.linear(x)                                        # (B, 4)
        direction = F.normalize(raw[:, :3], dim=1, eps=1e-8)             # unit sphere (B, 3)
        r         = SHELL_MIN_KM + torch.sigmoid(raw[:, 3]) * SHELL_RANGE_KM  # (B,) km
        xyz_km    = r.unsqueeze(1) * direction                            # (B, 3) km

        # Replicate MinMaxScaler: x_scaled = (x − min) / range
        return (xyz_km - self.scaler_min) / self.scaler_range             # (B, 3) in [0,1]

class MultiTaskMagneticNet(nn.Module):
    """
    Multi-task neural network for magnetic source localization and classification.

    Architecture:
    - Shared backbone: Processes magnetic field readings
    - Regression head: Predicts 3D source position (x, y, z)
    - Classification head: Predicts source type (monopole/dipole)

    Features:
    - Residual connections for better gradient flow
    - Batch normalization for stable training
    - Task-specific heads with proper output activations
    """

    def __init__(self, config: ModelConfig,
                 use_spherical_loc: bool = False,
                 loc_scaler_min=None, loc_scaler_range=None):
        super().__init__()
        self.config = config
        self.use_spherical_loc = use_spherical_loc

        # Activation function
        if config.activation == "relu":
            self.activation = nn.ReLU()
        elif config.activation == "silu":
            self.activation = nn.SiLU()
        elif config.activation == "gelu":
            self.activation = nn.GELU()
        else:
            raise ValueError(f"Unsupported activation: {config.activation}")

        # ========== Shared Backbone ==========
        layers = []
        prev_dim = config.input_dim

        for i, hidden_dim in enumerate(config.hidden_dims):
            if config.use_residual:
                # Projection shortcut handles any dimension change, so the
                # residual block can be applied to every layer unconditionally.
                layers.append(
                    ResidualBlock(prev_dim, hidden_dim, config.dropout_rate,
                                  config.activation, config.batch_norm_momentum)
                )
            else:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.BatchNorm1d(hidden_dim, momentum=config.batch_norm_momentum))
                layers.append(self.activation)
                layers.append(nn.Dropout(config.dropout_rate))

            prev_dim = hidden_dim

        self.shared_backbone = nn.Sequential(*layers)
        self.shared_dim = prev_dim

        # ========== Regression Head (Position) ==========
        loc_layers = []
        prev_dim = self.shared_dim

        for hidden_dim in config.loc_hidden_dims:
            loc_layers.append(nn.Linear(prev_dim, hidden_dim))
            loc_layers.append(nn.BatchNorm1d(hidden_dim, momentum=config.batch_norm_momentum))
            loc_layers.append(self._make_activation())  # fresh instance per layer
            loc_layers.append(nn.Dropout(config.dropout_rate * 0.5))  # less dropout for regression
            prev_dim = hidden_dim

        # Output layer: either constrained spherical or raw Cartesian
        if use_spherical_loc:
            # SphericalOutputLayer contains its own Linear(→3) +
            # physical parametrisation + MinMaxScaler conversion.
            loc_layers.append(
                SphericalOutputLayer(prev_dim, loc_scaler_min, loc_scaler_range)
            )
        else:
            loc_layers.append(nn.Linear(prev_dim, 3))
        self.regression_head = nn.Sequential(*loc_layers)

        # ========== Classification Head (Source Type) ==========
        type_layers = []
        prev_dim = self.shared_dim

        for hidden_dim in config.type_hidden_dims:
            type_layers.append(nn.Linear(prev_dim, hidden_dim))
            type_layers.append(nn.BatchNorm1d(hidden_dim, momentum=config.batch_norm_momentum))
            type_layers.append(self._make_activation())  # fresh instance per layer
            type_layers.append(nn.Dropout(config.dropout_rate))
            prev_dim = hidden_dim

        # Output layer - logits for 2 classes (monopole, dipole)
        type_layers.append(nn.Linear(prev_dim, 2))
        self.classification_head = nn.Sequential(*type_layers)

        # Initialize weights
        self._initialize_weights()

        # Re-initialise the spherical head after the global xavier(gain=0.5) sweep
        # so F.normalize sees moderate-size raw direction vectors.
        if use_spherical_loc:
            last = self.regression_head[-1]
            if isinstance(last, SphericalOutputLayer):
                last.reset_parameters()

    def _make_activation(self) -> nn.Module:
        """Return a fresh, independent activation instance.

        Each call produces a new object so no two layers in the network
        share the same module reference — avoiding inplace-operation
        conflicts and double-registration in PyTorch's module graph.
        """
        if self.config.activation == "relu":  return nn.ReLU()
        elif self.config.activation == "silu": return nn.SiLU()
        elif self.config.activation == "gelu": return nn.GELU()

    def _initialize_weights(self):
        """Initialize weights using Xavier initialization."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight, gain=0.5)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch_size, input_dim)

        Returns:
            loc_pred: Position predictions of shape (batch_size, 3)
            type_pred: Type logits of shape (batch_size, 2)
        """
        # Shared features
        features = self.shared_backbone(x)

        # Task-specific predictions
        loc_pred = self.regression_head(features)
        type_pred = self.classification_head(features)

        return loc_pred, type_pred


class MagneticSourceLoss(nn.Module):
    """
    Combined loss for multi-task learning.

    Uses:
    - Huber loss (smooth L1) for position regression (robust to outliers)
    - Cross-entropy loss for source type classification

    Adaptive weighting based on uncertainty (optional).
    """

    def __init__(self, loc_weight: float = 1.0, type_weight: float = 1.0,
                 use_uncertainty_weighting: bool = False, huber_beta: float = 0.1):
        """
        Args:
            loc_weight: Weight for position loss
            type_weight: Weight for classification loss
            use_uncertainty_weighting: Learn task weights automatically (Kendall et al. 2018)
        """
        super().__init__()
        self.loc_weight = loc_weight
        self.type_weight = type_weight
        self.use_uncertainty_weighting = use_uncertainty_weighting

        if use_uncertainty_weighting:
            # Learnable log variances for each task
            self.log_var_loc = nn.Parameter(torch.zeros(1))
            self.log_var_type = nn.Parameter(torch.zeros(1))

        self.loc_loss_fn = nn.SmoothL1Loss(beta=huber_beta)  # Huber loss
        self.type_loss_fn = nn.CrossEntropyLoss()

    def forward(self, loc_pred: torch.Tensor, type_pred: torch.Tensor,
                loc_true: torch.Tensor, type_true: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Compute combined loss.

        Args:
            loc_pred: Predicted positions (batch_size, 3)
            type_pred: Predicted type logits (batch_size, 2)
            loc_true: True positions (batch_size, 3)
            type_true: True type indices (batch_size,)

        Returns:
            Dictionary containing individual losses and total loss
        """
        # Position loss (regression)
        loc_loss = self.loc_loss_fn(loc_pred, loc_true)

        # Type loss (classification)
        type_loss = self.type_loss_fn(type_pred, type_true)

        if self.use_uncertainty_weighting:
            # Uncertainty-weighted multi-task loss
            # L = (1/2) * (L1 / exp(s1) + s1 + L2 / exp(s2) + s2)
            loc_weighted = 0.5 * (loc_loss / torch.exp(self.log_var_loc) + self.log_var_loc)
            type_weighted = 0.5 * (type_loss / torch.exp(self.log_var_type) + self.log_var_type)
            total_loss = loc_weighted + type_weighted
        else:
            total_loss = self.loc_weight * loc_loss + self.type_weight * type_loss

        return {
            'total_loss': total_loss,
            'loc_loss': loc_loss,
            'type_loss': type_loss,
            'loc_rmse': torch.sqrt(torch.mean((loc_pred - loc_true) ** 2)),
            'type_acc': (type_pred.argmax(dim=1) == type_true).float().mean()
        }


class EarlyStopping:
    """Early stopping to prevent overfitting."""

    def __init__(self, patience: int = 20, min_delta: float = 1e-5, mode: str = 'min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state_dict = None

    def __call__(self, score: float, model: nn.Module) -> bool:
        if self.best_score is None:
            self.best_score = score
            self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False

        if self.mode == 'min':
            improved = score < self.best_score - self.min_delta
        else:
            improved = score > self.best_score + self.min_delta

        if improved:
            self.best_score = score
            self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

        return self.early_stop


class ModelTrainer:
    """Industry-standard trainer with metrics tracking and checkpointing."""

    def __init__(self, model: nn.Module, device: torch.device, config: ModelConfig):
        self.model = model.to(device)
        self.device = device
        self.config = config

        # Optimizer with weight decay and momentum
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay,
            betas=(0.9, 0.999)
        )

        # Learning rate scheduler with warmup
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=10
        )

        self.criterion = MagneticSourceLoss(
            loc_weight=config.loc_loss_weight,
            type_weight=config.type_loss_weight
        )

        # Metrics tracking
        self.train_history = {'loss': [], 'loc_loss': [], 'type_loss': [], 'loc_rmse': [], 'type_acc': []}
        self.val_history = {'loss': [], 'loc_loss': [], 'type_loss': [], 'loc_rmse': [], 'type_acc': []}

    def train_epoch(self, train_loader: DataLoader) -> Dict[str, float]:
        """Train for one epoch."""
        self.model.train()
        epoch_metrics = {'loss': 0, 'loc_loss': 0, 'type_loss': 0, 'loc_rmse': 0, 'type_acc': 0}

        for X_batch, loc_batch, type_batch in train_loader:
            X_batch = X_batch.to(self.device)
            loc_batch = loc_batch.to(self.device)
            type_batch = type_batch.to(self.device)

            # Forward pass
            loc_pred, type_pred = self.model(X_batch)
            loss_dict = self.criterion(loc_pred, type_pred, loc_batch, type_batch)

            # Backward pass
            self.optimizer.zero_grad()
            loss_dict['total_loss'].backward()

            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()

            # Accumulate metrics
            for key in epoch_metrics.keys():
                # Loss dict exposes the combined loss as 'total_loss';
                # map our 'loss' accumulator onto it so the keys line up.
                src_key = 'total_loss' if key == 'loss' else key
                epoch_metrics[key] += loss_dict[src_key].item()

        # Average metrics
        n_batches = len(train_loader)
        return {k: v / n_batches for k, v in epoch_metrics.items()}

    @torch.no_grad()
    def evaluate(self, val_loader: DataLoader) -> Dict[str, float]:
        """Evaluate model on validation set."""
        self.model.eval()
        epoch_metrics = {'loss': 0, 'loc_loss': 0, 'type_loss': 0, 'loc_rmse': 0, 'type_acc': 0}

        for X_batch, loc_batch, type_batch in val_loader:
            X_batch = X_batch.to(self.device)
            loc_batch = loc_batch.to(self.device)
            type_batch = type_batch.to(self.device)

            loc_pred, type_pred = self.model(X_batch)
            loss_dict = self.criterion(loc_pred, type_pred, loc_batch, type_batch)

            for key in epoch_metrics.keys():
                # Loss dict exposes the combined loss as 'total_loss';
                # map our 'loss' accumulator onto it so the keys line up.
                src_key = 'total_loss' if key == 'loss' else key
                epoch_metrics[key] += loss_dict[src_key].item()

        n_batches = len(val_loader)
        return {k: v / n_batches for k, v in epoch_metrics.items()}

    def train(self, train_loader: DataLoader, val_loader: DataLoader,
              epochs: int = 200, early_stopping_patience: int = 30) -> Dict:
        """Complete training loop with early stopping."""
        early_stopping = EarlyStopping(patience=early_stopping_patience, mode='min')
        best_val_loss = float('inf')

        for epoch in range(epochs):
            # Train
            train_metrics = self.train_epoch(train_loader)
            for key, value in train_metrics.items():
                self.train_history[key].append(value)

            # Validate
            val_metrics = self.evaluate(val_loader)
            for key, value in val_metrics.items():
                self.val_history[key].append(value)

            # Learning rate scheduling
            self.scheduler.step(val_metrics['loss'])

            # Print progress
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | "
                      f"Train Loss: {train_metrics['loss']:.4f} | "
                      f"Val Loss: {val_metrics['loss']:.4f} | "
                      f"Val Loc RMSE: {val_metrics['loc_rmse']:.4f} | "
                      f"Val Type Acc: {val_metrics['type_acc']:.4f}")

            # Early stopping
            if early_stopping(val_metrics['loss'], self.model):
                print(f"Early stopping triggered at epoch {epoch+1}")
                self.model.load_state_dict(early_stopping.best_state_dict)
                break

        return {
            'train_history': self.train_history,
            'val_history': self.val_history,
            'best_val_loss': early_stopping.best_score if early_stopping.best_score else best_val_loss
        }

In [ ]:
class TunableTrainer(ModelTrainer):
    """
    ModelTrainer subclass with tunable grad_clip, loss, and scheduler.

    Overrides train_epoch() to use self.grad_clip instead of the
    hardcoded max_norm=1.0 in the base class. All other behaviour
    (early stopping, metrics tracking, evaluate()) is inherited.
    """

    def __init__(self, model: nn.Module, device: torch.device,
                 config: ModelConfig, grad_clip: float,
                 huber_beta: float, scheduler_factor: float):
        super().__init__(model, device, config)

        # Store tunable values
        self.grad_clip = grad_clip

        # Override criterion with tuned huber_beta
        self.criterion = MagneticSourceLoss(
            loc_weight  = config.loc_loss_weight,
            type_weight = config.type_loss_weight,
            huber_beta  = huber_beta,
        )

        # Override scheduler — also removes verbose=True (crashed PyTorch 2.4)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min',
            factor   = scheduler_factor,
            patience = 10,
        )

    def train_epoch(self, train_loader: DataLoader) -> Dict[str, float]:
        """Train one epoch using self.grad_clip instead of hardcoded 1.0."""
        self.model.train()
        epoch_metrics = {
            'loss': 0, 'loc_loss': 0, 'type_loss': 0,
            'loc_rmse': 0, 'type_acc': 0
        }

        for X_batch, loc_batch, type_batch in train_loader:
            X_batch    = X_batch.to(self.device)
            loc_batch  = loc_batch.to(self.device)
            type_batch = type_batch.to(self.device)

            loc_pred, type_pred = self.model(X_batch)
            loss_dict = self.criterion(loc_pred, type_pred, loc_batch, type_batch)

            self.optimizer.zero_grad()
            loss_dict['total_loss'].backward()

            # Use the tuned clip value — this is the fix vs. Experiments 1 & 2
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), max_norm=self.grad_clip
            )

            self.optimizer.step()

            for key in epoch_metrics.keys():
                src_key = 'total_loss' if key == 'loss' else key
                epoch_metrics[key] += loss_dict[src_key].item()

        n_batches = len(train_loader)
        return {k: v / n_batches for k, v in epoch_metrics.items()}

In [ ]:
import numpy as np
import torch
import os
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader

def scale_magnetic_axes(X_matrix, scalers=None, is_training=False):
    """Reshapes flat vectors to (N*29, 3) to apply a per-axis RobustScaler."""
    n_samples = X_matrix.shape[0]
    reshaped = X_matrix.reshape(-1, 3)
    if is_training:
        scalers = RobustScaler()
        scaled_axes = scalers.fit_transform(reshaped)
    else:
        scaled_axes = scalers.transform(reshaped)
    return scaled_axes.reshape(n_samples, 87), scalers

def create_dataloader(X_np, loc_np, type_np, batch_size=256, shuffle=False):
    """Build a DataLoader from numpy arrays."""
    X_tensor    = torch.tensor(X_np,    dtype=torch.float32)
    loc_tensor  = torch.tensor(loc_np,  dtype=torch.float32)
    type_tensor = torch.tensor(type_np, dtype=torch.long)
    dataset = TensorDataset(X_tensor, loc_tensor, type_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 2 · Dataset Generation

Reproduces the Experiment 3 datasets exactly (same seeds):
- **Training reference** (`X_noisy_dataset`, 64,000 samples) — used only for
  the Bayes-error nearest-neighbour index in Section 6.
- **Test set** (`X_clean_dataset`, ~11,294 samples, noiseless) — used for
  evaluation.

If `analysis_data.npz` exists (saved by a prior run), it is loaded instead
of regenerating (~5 min on CPU / ~1 min on GPU).


In [ ]:
if os.path.exists('analysis_data.npz'):
    print('Loading pre-saved datasets from analysis_data.npz ...')
    _d = np.load('analysis_data.npz')
    X_noisy_dataset = _d['X_noisy']
    y_noisy_dataset = _d['y_noisy']
    X_clean_dataset = _d['X_clean']
    y_clean_dataset = _d['y_clean']
    print(f'  Noisy train: {X_noisy_dataset.shape}')
    print(f'  Clean test : {X_clean_dataset.shape}')
else:
    print('Generating datasets (this takes ~1-5 minutes) ...')
    # ─────────────────────────────────────────────────────────────────────
    # Exact reproduction of Experiment 3 dataset (seed 42 / seed 123)
    # ─────────────────────────────────────────────────────────────────────
    import numpy as np
    from tqdm import tqdm # Optional: pip install tqdm for a progress bar
    import warnings

    # --- 1. Configuration & Initialization ---
    # Using the 29 Earth-scale sensor locations loaded
    EARTH_RADIUS = 6371.0

    config = SimulatorConfig(
        sensor_positions=sensors_xyz_locations,
        magnetic_constant=1e-7,
        default_moment_range=(1e12, 1e14), # Scale moments up drastically so the field isn't wiped out by 1/r^3 over 6000+ km
        noise_std=0.01, # Baseline noise (can be adjusted to represent nT)
    )

    sim = TimeSeriesSimulator(sensors_xyz_locations, config)

    # --- 2. Define the Global Source Grid ---
    # Instead of a flat 10x10 grid, we generate a shell of sources above the sensors
    # representing orbital or ionospheric phenomena (e.g., altitudes of 800km to 3100km)
    np.random.seed(42)
    n_grid_points = 3200        # Experiment 3: 3,200 unique positions (x10 noise -> 64,000 samples)

    # Define the boundaries of your shell
    min_alt = EARTH_RADIUS + 800
    max_alt = EARTH_RADIUS + 3100

    grid_positions = []
    for _ in range(n_grid_points):
        # Distribute uniformly on a sphere, then scale by altitude
        vec = np.random.randn(3)
        vec /= np.linalg.norm(vec)
        alt = np.random.uniform(min_alt, max_alt)
        grid_positions.append(vec * alt)
    grid_positions = np.array(grid_positions)

    # --- 3. Dataset Generation Loop ---
    pole_types = [SourceType.MONOPOLE, SourceType.DIPOLE]
    noise_realizations = 10    # halved from 20 -> 10 to trade correlated augmentation for spatial coverage
    samples = len(grid_positions) * len(pole_types) * noise_realizations

    X_noisy_dataset = []
    y_noisy_dataset = []

    print(f"\n⚙️ Initiating Generation: {len(grid_positions)} positions × {len(pole_types)} types × {noise_realizations} noise profiles")
    print(f"🎯 Target Samples: {samples}")

    # Wrap in tqdm for a progress bar if installed, otherwise just use standard loop
    for pos in tqdm(grid_positions, desc="Processing Grid"):
        for p_type in pole_types:
            # Generate a massive random moment vector suitable for planetary scales
            _, magnetic_moment = sim.generate_source()

            # 1. Calculate clean, noiseless fields
            clean_fields = sim.compute_fields(pos, magnetic_moment, source_type=p_type)

            # 2. One-hot encode the target: [is_mono, is_dipole]
            one_hot = [
                1.0 if p_type == SourceType.MONOPOLE else 0.0,
                1.0 if p_type == SourceType.DIPOLE else 0.0,
            ]

            # Target shape: [X, Y, Z, Mono, Dip]
            label = np.concatenate((pos, one_hot))

            # 3. Apply noise realizations
            for _ in range(noise_realizations):
                noisy_fields = sim.add_noise(clean_fields, NoiseModel.GAUSSIAN)
                X_noisy_dataset.append(noisy_fields)
                y_noisy_dataset.append(label)

    # --- 4. Formatting and Export ---
    X_noisy_dataset = np.array(X_noisy_dataset)
    y_noisy_dataset = np.array(y_noisy_dataset)

    print("\n✅ Train/Val Dataset Generation Complete.")
    print(f"📊 Features (X) Shape: {X_noisy_dataset.shape}  | (Samples × {sim.n_sensors} sensors * 3 axes)")
    print(f"🎯 Targets  (y) Shape: {y_noisy_dataset.shape}   | (Samples × [X, Y, Z, Mono, Dip])")

    # Use a separate seed for the test grid so a partial re-run of this cell
    # cannot cause train/test position overlap via shared RNG state.
    np.random.seed(123)
    n_clean_grid_positions = int(3 * samples / 34) # 15% test set
    clean_grid_positions = []
    for _ in range(n_clean_grid_positions):
        # Distribute uniformly on a sphere, then scale by altitude
        vec = np.random.randn(3)
        vec /= np.linalg.norm(vec)
        alt = np.random.uniform(min_alt, max_alt)
        clean_grid_positions.append(vec * alt)
    clean_grid_positions = np.array(clean_grid_positions)

    X_clean_dataset = []
    y_clean_dataset = []

    print(f"\n⚙️ Initiating Generation: {n_clean_grid_positions} positions × {len(pole_types)} types")
    print(f"🎯 Target Samples: {n_clean_grid_positions * len(pole_types)}")

    # Wrap in tqdm for a progress bar if installed, otherwise just use standard loop
    for pos in tqdm(clean_grid_positions, desc="Processing Grid"):
        for p_type in pole_types:
            # Generate a massive random moment vector suitable for planetary scales
            _, magnetic_moment = sim.generate_source()

            # 1. Calculate clean, noiseless fields
            clean_fields = sim.compute_fields(pos, magnetic_moment, source_type=p_type)

            # 2. One-hot encode the target: [is_mono, is_dipole]
            one_hot = [
                1.0 if p_type == SourceType.MONOPOLE else 0.0,
                1.0 if p_type == SourceType.DIPOLE else 0.0,
            ]

            # Target shape: [X, Y, Z, Mono, Dip]
            label = np.concatenate((pos, one_hot))

            # Append data
            X_clean_dataset.append(clean_fields)
            y_clean_dataset.append(label)

    X_clean_dataset = np.array(X_clean_dataset)
    y_clean_dataset = np.array(y_clean_dataset)

    print("\n✅ Test Dataset Generation Complete.")
    print(f"📊 Features (X) Shape: {X_clean_dataset.shape}  | (Clean Samples × {sim.n_sensors} sensors * 3 axes)")
    print(f"🎯 Targets  (y) Shape: {y_clean_dataset.shape}   | (Clean Samples × [X, Y, Z, Mono, Dip])")
    # Save for faster re-runs
    np.savez('analysis_data.npz',
             X_noisy=X_noisy_dataset, y_noisy=y_noisy_dataset,
             X_clean=X_clean_dataset, y_clean=y_clean_dataset)
    print('Saved analysis_data.npz')


## 3 · Per-Experiment Evaluation

All metrics use **3D Euclidean RMSE** throughout.  The `per_axis_rmse` field
(= `sqrt(mean(element-wise squared errors))`) is stored only to cross-check
against the value saved in each checkpoint at training time; it is not used
in any table or figure.


In [ ]:
def evaluate_experiment(ckpt_path, X_clean, y_clean, device):
    """Load checkpoint, run on test set, return metrics + raw arrays."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = ckpt['config']
    use_spherical = ckpt.get('use_spherical_loc', False)

    if use_spherical:
        model = MultiTaskMagneticNet(
            config,
            use_spherical_loc=True,
            loc_scaler_min=ckpt['loc_scaler_min'],
            loc_scaler_range=ckpt['loc_scaler_range'],
        )
    else:
        model = MultiTaskMagneticNet(config)

    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()

    axis_scaler   = ckpt['axis_scalers']
    target_scaler = ckpt['target_scaler']

    X_scaled, _ = scale_magnetic_axes(X_clean, scalers=axis_scaler, is_training=False)
    loc_true_km     = y_clean[:, :3]
    loc_true_scaled = target_scaler.transform(loc_true_km)
    type_true       = np.argmax(y_clean[:, 3:], axis=1)

    loader = create_dataloader(X_scaled, loc_true_scaled, type_true, batch_size=256, shuffle=False)
    loc_preds, logits_list = [], []
    with torch.no_grad():
        for X_b, _, _ in loader:
            lp, tp = model(X_b.to(device))
            loc_preds.append(lp.cpu().numpy())
            logits_list.append(tp.cpu().numpy())

    loc_pred_scaled = np.vstack(loc_preds)
    type_logits     = np.vstack(logits_list)
    type_pred       = np.argmax(type_logits, axis=1)
    loc_pred_km     = target_scaler.inverse_transform(loc_pred_scaled)

    dist_km    = np.linalg.norm(loc_pred_km - loc_true_km, axis=1)
    errors_sc  = loc_pred_scaled - loc_true_scaled
    rmse_scaled = float(np.sqrt(np.mean(np.sum(errors_sc ** 2, axis=1))))
    rmse_km     = float(np.sqrt(np.mean(dist_km ** 2)))
    accuracy    = float(np.mean(type_pred == type_true))

    return {
        'accuracy'       : accuracy,
        'rmse_scaled'    : rmse_scaled,
        'rmse_km'        : rmse_km,
        'mae_km'         : float(np.mean(dist_km)),
        'median_km'      : float(np.percentile(dist_km, 50)),
        'p90_km'         : float(np.percentile(dist_km, 90)),
        'per_axis_rmse'  : float(np.sqrt(np.mean(errors_sc ** 2))),
        'ckpt_per_axis'  : float(ckpt['test_loc_rmse']),
        'dist_km'        : dist_km,
        'loc_pred_km'    : loc_pred_km,
        'loc_true_km'    : loc_true_km,
        'loc_pred_scaled': loc_pred_scaled,
        'loc_true_scaled': loc_true_scaled,
        'type_pred'      : type_pred,
        'type_true'      : type_true,
        'axis_scaler'    : axis_scaler,
        'target_scaler'  : target_scaler,
        'use_spherical'  : use_spherical,
    }


In [ ]:
EXPERIMENTS = [
    ('Exp 1 (Broad search)', 'magnetic_model_experiment1.pt'),
    ('Exp 2 (Loss/optim)',   'magnetic_model_experiment2.pt'),
    ('Exp 3 (Cartesian)',    'magnetic_model_experiment3.pt'),
    ('Exp 4 (Spherical)',    'magnetic_model_experiment4.pt'),
]

results = {}
for name, path in EXPERIMENTS:
    if os.path.exists(path):
        print(f'Evaluating {name} ...')
        results[name] = evaluate_experiment(path, X_clean_dataset, y_clean_dataset, device)
        r = results[name]
        # Cross-check: recomputed per-axis RMSE should match checkpoint value
        match = abs(r['per_axis_rmse'] - r['ckpt_per_axis']) < 1e-4
        print(f'  Accuracy : {r["accuracy"]:.2%}')
        print(f'  3D RMSE  : {r["rmse_scaled"]:.4f} scaled  |  {r["rmse_km"]:,.0f} km')
        print(f'  Per-axis RMSE check: {r["per_axis_rmse"]:.4f} (recomputed) vs '
              f'{r["ckpt_per_axis"]:.4f} (checkpoint) -> {"OK" if match else "MISMATCH"}')
    else:
        print(f'{name}: {path} not found — upload checkpoint to include this row')

print(f'\nLoaded {len(results)} experiment(s)')


## 4 · Cross-Experiment Comparison

In [ ]:
print(f'\n{"Experiment":<24} | {"Acc (%)":>8} | {"3D RMSE (sc)":>12} | {"3D RMSE (km)":>13} | {"Median (km)":>11} | {"P90 (km)":>9}')
print('-' * 88)
for name, r in results.items():
    print(f'{name:<24} | {r["accuracy"]*100:>8.2f} | {r["rmse_scaled"]:>12.4f} | {r["rmse_km"]:>13,.0f} | {r["median_km"]:>11,.0f} | {r["p90_km"]:>9,.0f}')


## 5 · Visualizations

Four figure groups. All are parameterised over whatever experiments were loaded.


### 5.1 Error Distributions & CDF (all experiments)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

if not results:
    print('No results loaded — upload at least one checkpoint')
else:
    colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Position Error Distributions — All Experiments', fontsize=13)

    all_dists = np.concatenate([r['dist_km'] for r in results.values()])
    bins = np.linspace(0, np.percentile(all_dists, 97), 60)

    ax_hist, ax_cdf = axes
    for (name, r), c in zip(results.items(), colors):
        label = f'{name}  RMSE {r["rmse_km"]:,.0f} km'
        ax_hist.hist(r['dist_km'], bins=bins, alpha=0.55, density=True, color=c, label=label)
        ax_hist.axvline(r['rmse_km'], color=c, linestyle='--', linewidth=1.5)

        sorted_d = np.sort(r['dist_km'])
        cdf = np.arange(1, len(sorted_d)+1) / len(sorted_d)
        ax_cdf.plot(sorted_d, cdf * 100, linewidth=2, color=c, label=label)

    ax_hist.set_title('Error Density')
    ax_hist.set_xlabel('3D Position Error (km)')
    ax_hist.set_ylabel('Density')
    ax_hist.legend(fontsize=8)
    ax_hist.grid(True, alpha=0.3)

    ax_cdf.axhline(50, color='gray', linestyle=':', linewidth=1)
    ax_cdf.axhline(90, color='gray', linestyle=':', linewidth=1)
    ax_cdf.set_title('Cumulative Error')
    ax_cdf.set_xlabel('3D Position Error (km)')
    ax_cdf.set_ylabel('Cumulative % of Test Samples')
    ax_cdf.legend(fontsize=8)
    ax_cdf.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


### 5.2 Per-Axis Residuals

In [ ]:
AXIS_NAMES = ['X (km)', 'Y (km)', 'Z (km)']

# Show at most 2 experiments side-by-side (prefer Exp 3 and Exp 4)
plot_exps = [k for k in results if 'Exp 3' in k or 'Exp 4' in k] or list(results.keys())[:2]

if not plot_exps:
    print('No results to plot')
else:
    n_exp = len(plot_exps)
    fig, axes = plt.subplots(n_exp, 3, figsize=(15, 4.5 * n_exp))
    if n_exp == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle('Per-Axis Residuals (pred − true)', fontsize=13)

    for row_i, name in enumerate(plot_exps):
        r = results[name]
        resid_km = r['loc_pred_km'] - r['loc_true_km']
        for ax_i, ax_name in enumerate(AXIS_NAMES):
            ax = axes[row_i, ax_i]
            true_coord = r['loc_true_km'][:, ax_i]
            resid_coord = resid_km[:, ax_i]

            ax.scatter(true_coord, resid_coord, s=1.5, alpha=0.15, c='#3498db')

            # Running mean
            order = np.argsort(true_coord)
            window = max(1, len(true_coord) // 60)
            rm = np.convolve(resid_coord[order], np.ones(window)/window, mode='valid')
            x_rm = true_coord[order][window//2:window//2+len(rm)]
            ax.plot(x_rm, rm, color='#e74c3c', linewidth=1.5)

            ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
            rmse_ax = float(np.sqrt(np.mean(resid_coord**2)))
            ax.set_title(f'{name}\n{ax_name}  per-axis RMSE {rmse_ax:,.0f} km')
            ax.set_xlabel(f'True {ax_name}')
            ax.set_ylabel('Residual (km)')
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


### 5.3 3D Scatter: Predicted vs. True Positions

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

plot_exps_3d = [k for k in results if 'Exp 3' in k or 'Exp 4' in k] or list(results.keys())[:2]

if not plot_exps_3d:
    print('No results to plot')
else:
    n_exp = len(plot_exps_3d)
    fig = plt.figure(figsize=(8 * n_exp, 7))
    fig.suptitle('3D Predicted vs. True Source Positions (300 sample subset)', fontsize=13)

    for col_i, name in enumerate(plot_exps_3d):
        r = results[name]
        ax = fig.add_subplot(1, n_exp, col_i + 1, projection='3d')

        idx = np.random.choice(len(r['loc_true_km']), min(300, len(r['loc_true_km'])), replace=False)
        true_pts = r['loc_true_km'][idx]
        pred_pts = r['loc_pred_km'][idx]
        errs     = r['dist_km'][idx]

        sc = ax.scatter(true_pts[:, 0], true_pts[:, 1], true_pts[:, 2],
                        c=errs, cmap='YlOrRd', s=20, alpha=0.7, label='True')
        ax.scatter(pred_pts[:, 0], pred_pts[:, 1], pred_pts[:, 2],
                   c=errs, cmap='YlOrRd', s=15, alpha=0.4, marker='^')

        for t, p in zip(true_pts, pred_pts):
            ax.plot([t[0], p[0]], [t[1], p[1]], [t[2], p[2]],
                    color='gray', alpha=0.15, linewidth=0.5)

        plt.colorbar(sc, ax=ax, shrink=0.6, label='Error (km)')
        ax.set_title(f'{name}\n3D RMSE {r["rmse_km"]:,.0f} km')
        ax.set_xlabel('X (km)'); ax.set_ylabel('Y (km)'); ax.set_zlabel('Z (km)')

    plt.tight_layout()
    plt.show()


### 5.4 Error by Source Type

In [ ]:
# Use the best-performing loaded experiment
if results:
    best_name = min(results, key=lambda k: results[k]['rmse_km'])
    r = results[best_name]

    mono_mask  = (r['type_true'] == 0)
    dipole_mask = (r['type_true'] == 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Error by Source Type — {best_name}', fontsize=13)

    all_err = r['dist_km']
    bins = np.linspace(0, np.percentile(all_err, 97), 50)

    ax_hist, ax_cdf = axes
    for mask, label, color in [
        (mono_mask,   'Monopole', '#e74c3c'),
        (dipole_mask, 'Dipole',   '#3498db'),
    ]:
        errs = r['dist_km'][mask]
        ax_hist.hist(errs, bins=bins, alpha=0.55, density=True,
                     color=color, label=f'{label} (n={mask.sum():,})')
        ax_hist.axvline(np.sqrt(np.mean(errs**2)), color=color,
                        linestyle='--', linewidth=1.5,
                        label=f'RMSE {np.sqrt(np.mean(errs**2)):,.0f} km')

        sorted_e = np.sort(errs)
        cdf = np.arange(1, len(sorted_e)+1) / len(sorted_e)
        ax_cdf.plot(sorted_e, cdf * 100, linewidth=2, color=color,
                    label=f'{label}  P50={np.median(errs):,.0f} km')

    ax_hist.set_title('Error Density by Type')
    ax_hist.set_xlabel('3D Position Error (km)')
    ax_hist.set_ylabel('Density')
    ax_hist.legend(fontsize=8)
    ax_hist.grid(True, alpha=0.3)

    ax_cdf.axhline(50, color='gray', linestyle=':', linewidth=1)
    ax_cdf.axhline(90, color='gray', linestyle=':', linewidth=1)
    ax_cdf.set_title('CDF by Type')
    ax_cdf.set_xlabel('3D Position Error (km)')
    ax_cdf.set_ylabel('Cumulative %')
    ax_cdf.legend(fontsize=8)
    ax_cdf.grid(True, alpha=0.3)

    print(f'Classification accuracy — Monopole: {(r["type_pred"][mono_mask]==0).mean():.2%}')
    print(f'Classification accuracy — Dipole:   {(r["type_pred"][dipole_mask]==1).mean():.2%}')

    plt.tight_layout()
    plt.show()


## 6 · Bayes Error Estimation (Fixed)

**Why the prior implementation failed:** It searched noisy training inputs
against clean test inputs. Noise pushes every training sample into a unique
neighbourhood, so queries always land far from anything meaningful → proxy
RMSE ≈ shell radius ≈ completely uninformative.

**Fix — centroid approach:** Average the 10 noise realizations per (position,
type) configuration before building the nearest-neighbour index. Each centroid
approximates the noiseless signal for that configuration, so clean test queries
now find genuinely similar sources. The position gap between a test sample and
its nearest training centroid quantifies physical forward-model ambiguity —
the irreducible confusion the model cannot resolve.

The proxy is a **lower bound** on Bayes error, so the model RMSE should be
**above** the proxy. A negative gap flags a bug.


In [ ]:
from sklearn.neighbors import NearestNeighbors
import time

if not results:
    print('No experiment results loaded — run Section 3 first')
else:
    # Use Exp 3 scalers as the reference (most stable Cartesian baseline)
    ref_name = next((k for k in results if 'Exp 3' in k), list(results.keys())[0])
    ref_axis_scaler   = results[ref_name]['axis_scaler']
    ref_target_scaler = results[ref_name]['target_scaler']

    print('=' * 60)
    print('BAYES ERROR ESTIMATION — Centroid Nearest-Neighbour Proxy')
    print('=' * 60)

    # ── 1. Scale training data in the same space as test ──────────
    print('\nScaling training reference set ...')
    X_train_sc, _ = scale_magnetic_axes(
        X_noisy_dataset, scalers=ref_axis_scaler, is_training=False
    )

    # ── 2. Sanity-check dataset layout assumption ─────────────────
    # Layout: (pos_0 mono noise_0..9)(pos_0 di noise_0..9)(pos_1 ...)
    # So consecutive groups of 10 share (position, type).
    # within-group std should be << across-group std if noise is moderate.
    # Always 10 for the Exp 3 training reference generated in Section 2.
    # Exp 1/2 used 20 noise realizations but their checkpoints are evaluated
    # against this same Exp 3 reference — the Bayes floor is a property of
    # the sensor array and source distribution, not the specific model.
    # Always 10 for the Exp 3 training reference generated in Section 2.
    # Exp 1/2 used 20 noise realizations but their checkpoints are evaluated
    # against this same Exp 3 reference — the Bayes floor is a property of
    # the sensor array and source distribution, not the specific model.
    noise_realizations = 10
    n_configs = len(X_train_sc) // noise_realizations
    X_grouped  = X_train_sc.reshape(n_configs, noise_realizations, 87)
    within_std = float(X_grouped.std(axis=1).mean())
    across_std = float(X_train_sc[::noise_realizations].std(axis=0).mean())
    print(f'  Layout check — within-group std: {within_std:.4f}, '
          f'across-group std: {across_std:.4f}')
    assert within_std < across_std, (
        'Layout assumption violated — check noise_realizations stride. '
        f'within={within_std:.4f} >= across={across_std:.4f}'
    )
    print('  Layout check passed.')

    # ── 3. Build centroids (average noise draws per config) ───────
    X_centroids   = X_grouped.mean(axis=1)                   # (n_configs, 87)
    loc_centroids = y_noisy_dataset[::noise_realizations, :3] # (n_configs, 3) raw km
    print(f'\n  Centroids: {X_centroids.shape[0]:,} unique (position, type) configs')

    # ── 4. Fit NN index on centroids ─────────────────────────────
    print('  Fitting nearest-neighbour index ...')
    t0 = time.time()
    nn_model = NearestNeighbors(
        n_neighbors=1, algorithm='ball_tree', metric='euclidean', n_jobs=-1
    ).fit(X_centroids)
    print(f'  Fit time: {time.time()-t0:.1f}s')

    # ── 5. Query with Exp 3 test inputs (noiseless) ───────────────
    # Use the test set scaled under the reference scaler
    ref_r = results[ref_name]
    X_test_sc   = ref_r['loc_pred_scaled'] * 0   # just get shape
    X_test_sc, _ = scale_magnetic_axes(
        X_clean_dataset, scalers=ref_axis_scaler, is_training=False
    )
    loc_test_raw    = ref_r['loc_true_km']
    loc_test_scaled = ref_r['loc_true_scaled']

    print('  Querying nearest neighbours ...')
    t0 = time.time()
    _, nn_idxs = nn_model.kneighbors(X_test_sc)
    print(f'  Query time: {time.time()-t0:.1f}s')

    # ── 6. Bayes proxy errors ─────────────────────────────────────
    nn_pos_km  = loc_centroids[nn_idxs[:, 0]]
    bayes_km   = np.linalg.norm(loc_test_raw - nn_pos_km, axis=1)
    nn_pos_sc  = ref_target_scaler.transform(nn_pos_km)
    bayes_sc   = np.sqrt(np.sum((loc_test_scaled - nn_pos_sc) ** 2, axis=1))
    bayes_rmse_scaled = float(np.sqrt(np.mean(bayes_sc ** 2)))
    bayes_rmse_km     = float(np.sqrt(np.mean(bayes_km ** 2)))
    bayes_p50_km      = float(np.percentile(bayes_km, 50))
    bayes_p90_km      = float(np.percentile(bayes_km, 90))

    print('\n' + '=' * 60)
    print('BAYES PROXY RESULTS')
    print('=' * 60)
    print(f'  3D RMSE (scaled) : {bayes_rmse_scaled:.4f}')
    print(f'  3D RMSE (km)     : {bayes_rmse_km:,.1f}')
    print(f'  Median (km)      : {bayes_p50_km:,.1f}')
    print(f'  P90 (km)         : {bayes_p90_km:,.1f}')

    print('\n  Per-experiment gap above Bayes proxy:')
    print(f'  {"Experiment":<24} | {"3D RMSE (sc)":>12} | {"Gap":>8} | {"Gap %":>8} | Interpretation')
    print('  ' + '-' * 80)
    for name, r in results.items():
        gap = r['rmse_scaled'] - bayes_rmse_scaled
        gap_pct = (gap / bayes_rmse_scaled) * 100
        if gap < 0:
            interp = 'PROXY TOO HIGH (fix needed)'
        elif gap_pct < 10:
            interp = 'Near-optimal'
        elif gap_pct < 30:
            interp = 'Strong'
        else:
            interp = 'Room to improve'
        print(f'  {name:<24} | {r["rmse_scaled"]:>12.4f} | {gap:>+8.4f} | {gap_pct:>7.1f}% | {interp}')

    # ── 7. Visualization ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Bayes Error Proxy — Centroid Nearest-Neighbour Method', fontsize=13)

    combined_max = np.percentile(
        np.concatenate([bayes_km] + [r['dist_km'] for r in results.values()]), 97
    )
    bins = np.linspace(0, combined_max, 60)

    ax_hist, ax_cdf = axes
    ax_hist.hist(bayes_km, bins=bins, alpha=0.6, color='#2ecc71', density=True,
                 label=f'Bayes proxy  RMSE {bayes_rmse_km:,.0f} km')
    ax_hist.axvline(bayes_rmse_km, color='#27ae60', linestyle='--', linewidth=2)

    colors = ['#3498db', '#e67e22', '#e74c3c', '#9b59b6']
    for (name, r), c in zip(results.items(), colors):
        ax_hist.hist(r['dist_km'], bins=bins, alpha=0.4, color=c, density=True,
                     label=f'{name}  RMSE {r["rmse_km"]:,.0f} km')
        ax_hist.axvline(r['rmse_km'], color=c, linestyle=':', linewidth=1.5)

    ax_hist.set_title('Error Density vs. Bayes Proxy')
    ax_hist.set_xlabel('3D Position Error (km)')
    ax_hist.set_ylabel('Density')
    ax_hist.legend(fontsize=7)
    ax_hist.grid(True, alpha=0.3)

    # CDF
    for arr, label, color in [
        (bayes_km, f'Bayes proxy  RMSE {bayes_rmse_km:,.0f} km', '#2ecc71'),
    ] + [(r['dist_km'], f'{n}  RMSE {r["rmse_km"]:,.0f} km', c)
         for (n, r), c in zip(results.items(), colors)]:
        s = np.sort(arr)
        ax_cdf.plot(s, np.arange(1, len(s)+1)/len(s)*100,
                    linewidth=2 if 'proxy' in label else 1.5,
                    linestyle='-' if 'proxy' in label else '--',
                    label=label, color=color)

    ax_cdf.axhline(50, color='gray', linestyle=':', linewidth=1)
    ax_cdf.axhline(90, color='gray', linestyle=':', linewidth=1)
    ax_cdf.set_title('CDF Comparison')
    ax_cdf.set_xlabel('3D Position Error (km)')
    ax_cdf.set_ylabel('Cumulative %')
    ax_cdf.legend(fontsize=7)
    ax_cdf.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 7 · Analysis & Conclusions

*(Fill in after all experiments have been evaluated.)*

Key questions to address:
1. Does constraining outputs to the spherical shell (Exp 4) improve over
   unconstrained Cartesian outputs (Exp 3)? By how much in 3D RMSE and km?
2. How large is the gap above the Bayes proxy floor? Is the model close to
   the theoretical ceiling for this sensor array and noise level?
3. Does position error vary systematically by altitude band or source type?
